In [1]:
print('hello world')

hello world


In [2]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from joblib import Parallel, delayed
import matplotlib.pyplot as plt 
import seaborn as sns
%matplotlib inline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
import scipy.stats as stats
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer, LabelEncoder,PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [3]:
df=pd.read_csv('agronimy mandi dataset.csv')
des_df=pd.read_csv('des dataset.csv')
apy=pd.read_csv('APY.csv')
apy_2024_demo=pd.read_csv('2024_demo.csv')
apy_2025_demo=pd.read_csv('2025_demo.csv')

In [4]:
print("Mandi states:")
print(sorted(df["State"].dropna().unique()))

print("\n2024 Demo states:")
print(sorted(apy_2024_demo["State"].dropna().unique()))

print("\n2025 Demo states:")
print(sorted(apy_2025_demo["State"].dropna().unique()))

Mandi states:
['Andhra Pradesh', 'Gujarat', 'Kerala', 'Madhya Pradesh', 'Punjab', 'Rajasthan', 'Uttar Pradesh', 'West Bengal']

2024 Demo states:
['Andaman and Nicobar Island', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'CHANDIGARH', 'Chhattisgarh', 'Dadra and Nagar Haveli', 'Daman and Diu', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu and Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']

2025 Demo states:
['Andaman and Nicobar Island', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'CHANDIGARH', 'Chhattisgarh', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu and Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Laddak', 'Madhya Pradesh', 'Maharashtra', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', '

In [5]:
# States present in ALL THREE datasets

mandi_states = set(df["State"].dropna().str.strip())
demo24_states = set(apy_2024_demo["State"].dropna().str.strip())
demo25_states = set(apy_2025_demo["State"].dropna().str.strip())

common_states = sorted(
    mandi_states & demo24_states & demo25_states
)

print("States common to all 3 datasets:", len(common_states))
print(common_states)

States common to all 3 datasets: 8
['Andhra Pradesh', 'Gujarat', 'Kerala', 'Madhya Pradesh', 'Punjab', 'Rajasthan', 'Uttar Pradesh', 'West Bengal']


In [6]:
# Create normalized State and District columns

mandi_locations = (
    df[["State", "District Name"]]
    .drop_duplicates()
    .copy()
)

mandi_locations["State_clean"] = (
    mandi_locations["State"]
    .astype(str)
    .str.strip()
    .str.lower()
)

mandi_locations["District_clean"] = (
    mandi_locations["District Name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

demo24_locations = (
    apy_2024_demo[["State", "District "]]
    .drop_duplicates()
    .copy()
)

demo24_locations["State_clean"] = (
    demo24_locations["State"]
    .astype(str)
    .str.strip()
    .str.lower()
)

demo24_locations["District_clean"] = (
    demo24_locations["District "]
    .astype(str)
    .str.strip()
    .str.lower()
)

demo25_locations = (
    apy_2025_demo[["State", "District "]]
    .drop_duplicates()
    .copy()
)

demo25_locations["State_clean"] = (
    demo25_locations["State"]
    .astype(str)
    .str.strip()
    .str.lower()
)

demo25_locations["District_clean"] = (
    demo25_locations["District "]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Find locations present in all three datasets
mandi_set = set(
    zip(mandi_locations["State_clean"], mandi_locations["District_clean"])
)

demo24_set = set(
    zip(demo24_locations["State_clean"], demo24_locations["District_clean"])
)

demo25_set = set(
    zip(demo25_locations["State_clean"], demo25_locations["District_clean"])
)

common_locations = sorted(
    mandi_set & demo24_set & demo25_set
)

print("Mandi locations:", len(mandi_set))
print("2024 Demo locations:", len(demo24_set))
print("2025 Demo locations:", len(demo25_set))
print("Common locations in ALL THREE:", len(common_locations))

print("\nFirst 50 common locations:")
for state, district in common_locations[:50]:
    print(state, "—", district)

Mandi locations: 270
2024 Demo locations: 693
2025 Demo locations: 700
Common locations in ALL THREE: 197

First 50 common locations:
andhra pradesh — anantapur
andhra pradesh — east godavari
andhra pradesh — guntur
andhra pradesh — krishna
andhra pradesh — kurnool
andhra pradesh — west godavari
gujarat — amreli
gujarat — anand
gujarat — bharuch
gujarat — bhavnagar
gujarat — botad
gujarat — dang
gujarat — devbhumi dwarka
gujarat — gandhinagar
gujarat — gir somnath
gujarat — jamnagar
gujarat — kachchh
gujarat — kheda
gujarat — morbi
gujarat — narmada
gujarat — navsari
gujarat — patan
gujarat — porbandar
gujarat — rajkot
gujarat — surat
gujarat — surendranagar
gujarat — valsad
kerala — alappuzha
kerala — ernakulam
kerala — idukki
kerala — kannur
kerala — kollam
kerala — kottayam
kerala — malappuram
kerala — pathanamthitta
kerala — thiruvananthapuram
kerala — wayanad
madhya pradesh — agar malwa
madhya pradesh — alirajpur
madhya pradesh — ashoknagar
madhya pradesh — balaghat
madhya pradesh

In [7]:
# ---------- MANDI ----------

mandi_sdc = df[["State", "District Name", "Commodity"]].drop_duplicates().copy()

mandi_sdc["State_clean"] = (
    mandi_sdc["State"].astype(str).str.strip().str.lower()
)

mandi_sdc["District_clean"] = (
    mandi_sdc["District Name"].astype(str).str.strip().str.lower()
)

mandi_sdc["Crop_clean"] = (
    mandi_sdc["Commodity"].astype(str).str.strip().str.lower()
)


# ---------- APY 2024 DEMO ----------

demo24_sdc = apy_2024_demo[["State", "District ", "Crop"]].drop_duplicates().copy()

demo24_sdc["State_clean"] = (
    demo24_sdc["State"].astype(str).str.strip().str.lower()
)

demo24_sdc["District_clean"] = (
    demo24_sdc["District "].astype(str).str.strip().str.lower()
)

demo24_sdc["Crop_clean"] = (
    demo24_sdc["Crop"].astype(str).str.strip().str.lower()
)


# ---------- APY 2025 DEMO ----------

demo25_sdc = apy_2025_demo[["State", "District ", "Crop"]].drop_duplicates().copy()

demo25_sdc["State_clean"] = (
    demo25_sdc["State"].astype(str).str.strip().str.lower()
)

demo25_sdc["District_clean"] = (
    demo25_sdc["District "].astype(str).str.strip().str.lower()
)

demo25_sdc["Crop_clean"] = (
    demo25_sdc["Crop"].astype(str).str.strip().str.lower()
)


# ---------- NORMALIZE OBVIOUS CROP NAME DIFFERENCES ----------

crop_mapping = {
    "bajra(pearl millet/cumbu)": "bajra",
    "jowar(sorghum)": "jowar",
    "lentil (masur)(whole)": "lentil",
    "soyabean": "soybean",
    "green gram (moong)(whole)": "moong",
    "arhar (tur/red gram)(whole)": "tur",
    "mustard": "rapeseed & mustard"
}

mandi_sdc["Crop_clean"] = mandi_sdc["Crop_clean"].replace(crop_mapping)


# ---------- CREATE SETS ----------

mandi_set = set(zip(
    mandi_sdc["State_clean"],
    mandi_sdc["District_clean"],
    mandi_sdc["Crop_clean"]
))

demo24_set = set(zip(
    demo24_sdc["State_clean"],
    demo24_sdc["District_clean"],
    demo24_sdc["Crop_clean"]
))

demo25_set = set(zip(
    demo25_sdc["State_clean"],
    demo25_sdc["District_clean"],
    demo25_sdc["Crop_clean"]
))


# ---------- COMMON TO ALL THREE ----------

common_sdc = sorted(
    mandi_set & demo24_set & demo25_set
)

print("Mandi State-District-Crop combinations:", len(mandi_set))
print("2024 Demo combinations:", len(demo24_set))
print("2025 Demo combinations:", len(demo25_set))
print("Common combinations in ALL THREE:", len(common_sdc))

print("\nFirst 50:")
for state, district, crop in common_sdc[:50]:
    print(f"{state} | {district} | {crop}")

Mandi State-District-Crop combinations: 2970
2024 Demo combinations: 15360
2025 Demo combinations: 15822
Common combinations in ALL THREE: 532

First 50:
andhra pradesh | anantapur | groundnut
andhra pradesh | anantapur | maize
andhra pradesh | krishna | maize
andhra pradesh | kurnool | bajra
andhra pradesh | kurnool | groundnut
andhra pradesh | kurnool | jowar
andhra pradesh | kurnool | maize
andhra pradesh | west godavari | maize
gujarat | amreli | bajra
gujarat | amreli | garlic
gujarat | amreli | groundnut
gujarat | amreli | jowar
gujarat | amreli | maize
gujarat | anand | bajra
gujarat | anand | maize
gujarat | bharuch | bajra
gujarat | bharuch | jowar
gujarat | bharuch | maize
gujarat | bhavnagar | bajra
gujarat | bhavnagar | garlic
gujarat | bhavnagar | groundnut
gujarat | bhavnagar | maize
gujarat | botad | bajra
gujarat | botad | groundnut
gujarat | devbhumi dwarka | bajra
gujarat | devbhumi dwarka | groundnut
gujarat | gandhinagar | bajra
gujarat | gir somnath | bajra
gujarat

In [8]:
# Count unique mandi trading days for every State-District-Crop combination

mandi_coverage = (
    df.groupby(
        ["State", "District Name", "Commodity"]
    )["Price Date"]
    .nunique()
    .reset_index(name="unique_days")
)

# Create the same normalized keys
mandi_coverage["State_clean"] = (
    mandi_coverage["State"].astype(str).str.strip().str.lower()
)

mandi_coverage["District_clean"] = (
    mandi_coverage["District Name"].astype(str).str.strip().str.lower()
)

mandi_coverage["Crop_clean"] = (
    mandi_coverage["Commodity"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(crop_mapping)
)

# Keep only combinations common to all 3 datasets
common_df = pd.DataFrame(
    common_sdc,
    columns=["State_clean", "District_clean", "Crop_clean"]
)

ranked_candidates = common_df.merge(
    mandi_coverage[
        [
            "State_clean",
            "District_clean",
            "Crop_clean",
            "State",
            "District Name",
            "Commodity",
            "unique_days"
        ]
    ],
    on=["State_clean", "District_clean", "Crop_clean"],
    how="left"
)

# Rank by number of available mandi days
ranked_candidates = ranked_candidates.sort_values(
    "unique_days",
    ascending=False
).reset_index(drop=True)

print("Total common candidates:", len(ranked_candidates))

print("\nTop 30 candidates:")
display(
    ranked_candidates[
        ["State", "District Name", "Commodity", "unique_days"]
    ].head(30)
)

Total common candidates: 532

Top 30 candidates:


,State,District Name,Commodity,unique_days
0,West Bengal,Bankura,Wheat,365
1,Uttar Pradesh,Mainpuri,Wheat,365
2,West Bengal,Birbhum,Wheat,364
3,Uttar Pradesh,Fatehpur,Wheat,364
4,Uttar Pradesh,Gonda,Banana,362
5,Uttar Pradesh,Hardoi,Wheat,362
6,Uttar Pradesh,Shahjahanpur,Wheat,362
7,West Bengal,Paschim Bardhaman,Wheat,362
8,Uttar Pradesh,Balrampur,Wheat,360
9,Uttar Pradesh,Shamli,Garlic,358


In [9]:
# Take top 25 mandi candidates
top_candidates = ranked_candidates.head(25).copy()

# Function to count APY records for a State-District-Crop
def get_apy_count(apy_data, state, district, crop):
    temp = apy_data.copy()

    temp["State_clean"] = (
        temp["State"].astype(str).str.strip().str.lower()
    )
    temp["District_clean"] = (
        temp["District "].astype(str).str.strip().str.lower()
    )
    temp["Crop_clean"] = (
        temp["Crop"].astype(str).str.strip().str.lower()
    )

    return len(
        temp[
            (temp["State_clean"] == state) &
            (temp["District_clean"] == district) &
            (temp["Crop_clean"] == crop)
        ]
    )

# Calculate APY record counts
top_candidates["demo24_records"] = top_candidates.apply(
    lambda row: get_apy_count(
        apy_2024_demo,
        row["State_clean"],
        row["District_clean"],
        row["Crop_clean"]
    ),
    axis=1
)

top_candidates["demo25_records"] = top_candidates.apply(
    lambda row: get_apy_count(
        apy_2025_demo,
        row["State_clean"],
        row["District_clean"],
        row["Crop_clean"]
    ),
    axis=1
)

display(
    top_candidates[
        [
            "State",
            "District Name",
            "Commodity",
            "unique_days",
            "demo24_records",
            "demo25_records"
        ]
    ]
)

,State,District Name,Commodity,unique_days,demo24_records,demo25_records
0,West Bengal,Bankura,Wheat,365,1,1
1,Uttar Pradesh,Mainpuri,Wheat,365,1,1
2,West Bengal,Birbhum,Wheat,364,1,1
3,Uttar Pradesh,Fatehpur,Wheat,364,1,1
4,Uttar Pradesh,Gonda,Banana,362,1,1
5,Uttar Pradesh,Hardoi,Wheat,362,1,1
6,Uttar Pradesh,Shahjahanpur,Wheat,362,1,1
7,West Bengal,Paschim Bardhaman,Wheat,362,1,1
8,Uttar Pradesh,Balrampur,Wheat,360,1,1
9,Uttar Pradesh,Shamli,Garlic,358,1,1


In [10]:
# Inspect APY records for the top 4 candidates

candidates_to_check = [
    ("West Bengal", "Bankura", "Wheat"),
    ("Uttar Pradesh", "Mainpuri", "Wheat"),
    ("West Bengal", "Birbhum", "Wheat"),
    ("Uttar Pradesh", "Fatehpur", "Wheat")
]

for state, district, crop in candidates_to_check:
    print("\n" + "=" * 70)
    print(f"{state} | {district} | {crop}")
    print("=" * 70)

    print("\n2024 Demo:")
    display(
        apy_2024_demo[
            (apy_2024_demo["State"].str.strip().str.lower() == state.lower()) &
            (apy_2024_demo["District "].str.strip().str.lower() == district.lower()) &
            (apy_2024_demo["Crop"].str.strip().str.lower() == crop.lower())
        ]
    )

    print("\n2025 Demo:")
    display(
        apy_2025_demo[
            (apy_2025_demo["State"].str.strip().str.lower() == state.lower()) &
            (apy_2025_demo["District "].str.strip().str.lower() == district.lower()) &
            (apy_2025_demo["Crop"].str.strip().str.lower() == crop.lower())
        ]
    )


West Bengal | Bankura | Wheat

2024 Demo:


,State,District,Crop,Crop_Year,Season,Area,Production,Yield,Demo_Year
18284,West Bengal,BANKURA,Wheat,2018,Rabi,2254.0,5788.0,2.57,2024



2025 Demo:


,State,District,Crop,Crop_Year,Season,Area,Production,Yield,Demo_Year
19246,West Bengal,BANKURA,Wheat,2019,Rabi,1986.0,3678.0,1.85,2025



Uttar Pradesh | Mainpuri | Wheat

2024 Demo:


...

,State,District,Crop,Crop_Year,Season,Area,Production,Yield,Demo_Year
19247,West Bengal,BIRBHUM,Wheat,2019,Rabi,27607.0,82865.0,3.0,2025



Uttar Pradesh | Fatehpur | Wheat

2024 Demo:


,State,District,Crop,Crop_Year,Season,Area,Production,Yield,Demo_Year
17351,Uttar Pradesh,FATEHPUR,Wheat,2018,Rabi,172432.0,687681.0,3.99,2024



2025 Demo:


,State,District,Crop,Crop_Year,Season,Area,Production,Yield,Demo_Year
18324,Uttar Pradesh,FATEHPUR,Wheat,2019,Rabi,179472.0,710602.0,3.96,2025


In [11]:
temp = df[
    (df["State"].str.strip().str.lower() == "west bengal") &
    (df["District Name"].str.strip().str.lower() == "bankura") &
    (df["Commodity"].str.strip().str.lower() == "wheat")
]

print(temp["Market Name"].value_counts())

Market Name
Bishnupur(Bankura)    364
Khatra                314
Name: count, dtype: int64


In [12]:
series = df[
    (df["State"].str.strip().str.lower() == "west bengal") &
    (df["District Name"].str.strip().str.lower() == "bankura") &
    (df["Market Name"].str.strip().str.lower() == "bishnupur(bankura)") &
    (df["Commodity"].str.strip().str.lower() == "wheat")
].copy()

print("Rows:", len(series))
print("Unique dates:", series["Price Date"].nunique())
print("Date range:", series["Price Date"].min(), "to", series["Price Date"].max())

print("\nDuplicate dates:")
print(series["Price Date"].duplicated().sum())

print("\nMissing dates:")
all_dates = pd.date_range(
    series["Price Date"].min(),
    series["Price Date"].max(),
    freq="D"
)

missing_dates = all_dates.difference(series["Price Date"])

print("Number of missing calendar dates:", len(missing_dates))
print("First missing dates:", missing_dates[:20].tolist())

Rows: 364
Unique dates: 364
Date range: 01 Apr 2025 to 31 Oct 2024

Duplicate dates:
0

Missing dates:
Number of missing calendar dates: 0
First missing dates: []


In [13]:
master_df = series[
    [
        "State",
        "District Name",
        "Market Name",
        "Commodity",
        "Price Date",
        "Min Price (Rs./Quintal)",
        "Max Price (Rs./Quintal)",
        "Modal Price (Rs./Quintal)"
    ]
].copy()

master_df = master_df.sort_values("Price Date").reset_index(drop=True)

print("Shape:", master_df.shape)
display(master_df.head())
display(master_df.tail())

Shape: (364, 8)


,State,District Name,Market Name,Commodity,Price Date,Min Price (Rs./Quintal),Max Price (Rs./Quintal),Modal Price (Rs./Quintal)
0,West Bengal,Bankura,Bishnupur(Bankura),Wheat,01 Apr 2025,2800.0,3100.0,2900.0
1,West Bengal,Bankura,Bishnupur(Bankura),Wheat,01 Aug 2025,2800.0,3200.0,3000.0
2,West Bengal,Bankura,Bishnupur(Bankura),Wheat,01 Dec 2024,2600.0,2800.0,2700.0
3,West Bengal,Bankura,Bishnupur(Bankura),Wheat,01 Feb 2025,2700.0,3000.0,2800.0
4,West Bengal,Bankura,Bishnupur(Bankura),Wheat,01 Jan 2025,2750.0,3000.0,2850.0


,State,District Name,Market Name,Commodity,Price Date,Min Price (Rs./Quintal),Max Price (Rs./Quintal),Modal Price (Rs./Quintal)
359,West Bengal,Bankura,Bishnupur(Bankura),Wheat,31 Jan 2025,2700.0,3000.0,2800.0
360,West Bengal,Bankura,Bishnupur(Bankura),Wheat,31 Jul 2025,2800.0,3200.0,3000.0
361,West Bengal,Bankura,Bishnupur(Bankura),Wheat,31 Mar 2025,2800.0,3100.0,2900.0
362,West Bengal,Bankura,Bishnupur(Bankura),Wheat,31 May 2025,2800.0,3200.0,3000.0
363,West Bengal,Bankura,Bishnupur(Bankura),Wheat,31 Oct 2024,2700.0,3000.0,2800.0


In [14]:
master_df.isnull().sum()
df.isnull().sum()

Sl no.                       0
District Name                0
Market Name                  0
Commodity                    0
Variety                      0
Grade                        0
Min Price (Rs./Quintal)      0
Max Price (Rs./Quintal)      0
Modal Price (Rs./Quintal)    0
Price Date                   0
State                        0
dtype: int64

In [15]:
master_df["Price Date"] = pd.to_datetime(
    master_df["Price Date"],
    errors="coerce"
)

print(master_df["Price Date"].dtype)
print(master_df["Price Date"].isna().sum())

datetime64[us]
0


In [16]:
master_df["Year"] = master_df["Price Date"].dt.year
master_df["Month"] = master_df["Price Date"].dt.month
master_df["Month_Name"] = master_df["Price Date"].dt.month_name()
master_df["Quarter"] = master_df["Price Date"].dt.quarter
master_df["Day_of_Week"] = master_df["Price Date"].dt.dayofweek
master_df["Day_Name"] = master_df["Price Date"].dt.day_name()
master_df["Day_of_Year"] = master_df["Price Date"].dt.dayofyear

print(master_df[
    [
        "Price Date",
        "Year",
        "Month",
        "Month_Name",
        "Quarter",
        "Day_of_Week",
        "Day_Name",
        "Day_of_Year"
    ]
].head(10))

  Price Date  Year  Month Month_Name  Quarter  Day_of_Week   Day_Name  \
0 2025-04-01  2025      4      April        2            1    Tuesday   
1 2025-08-01  2025      8     August        3            4     Friday   
2 2024-12-01  2024     12   December        4            6     Sunday   
3 2025-02-01  2025      2   February        1            5   Saturday   
4 2025-01-01  2025      1    January        1            2  Wednesday   
5 2025-07-01  2025      7       July        3            1    Tuesday   
6 2025-06-01  2025      6       June        2            6     Sunday   
7 2025-03-01  2025      3      March        1            5   Saturday   
8 2025-05-01  2025      5        May        2            3   Thursday   
9 2024-11-01  2024     11   November        4            4     Friday   

   Day_of_Year  
0           91  
1          213  
2          336  
3           32  
4            1  
5          182  
6          152  
7           60  
8          121  
9          306  


In [17]:
def get_price_season(month):
    if month in [6, 7, 8, 9, 10]:
        return "Kharif"
    elif month in [11, 12, 1, 2]:
        return "Rabi"
    else:
        return "Summer"


master_df["Price_Season"] = master_df["Month"].apply(get_price_season)

# Wheat's agricultural season according to our APY data
master_df["Crop_Season"] = "Rabi"

print(
    master_df[
        ["Price Date", "Year", "Month", "Price_Season", "Crop_Season"]
    ].head(20)
)

print("\nPrice season distribution:")
print(master_df["Price_Season"].value_counts())

   Price Date  Year  Month Price_Season Crop_Season
0  2025-04-01  2025      4       Summer        Rabi
1  2025-08-01  2025      8       Kharif        Rabi
2  2024-12-01  2024     12         Rabi        Rabi
3  2025-02-01  2025      2         Rabi        Rabi
4  2025-01-01  2025      1         Rabi        Rabi
5  2025-07-01  2025      7       Kharif        Rabi
6  2025-06-01  2025      6       Kharif        Rabi
7  2025-03-01  2025      3       Summer        Rabi
8  2025-05-01  2025      5       Summer        Rabi
9  2024-11-01  2024     11         Rabi        Rabi
10 2024-10-01  2024     10       Kharif        Rabi
11 2024-09-01  2024      9       Kharif        Rabi
12 2025-04-02  2025      4       Summer        Rabi
13 2025-08-02  2025      8       Kharif        Rabi
14 2024-12-02  2024     12         Rabi        Rabi
15 2025-02-02  2025      2         Rabi        Rabi
16 2025-01-02  2025      1         Rabi        Rabi
17 2025-07-02  2025      7       Kharif        Rabi
18 2025-06-0

In [18]:
# Get the APY information for our selected State + District + Crop

apy_24 = apy_2024_demo[
    (apy_2024_demo["State"].str.strip().str.lower() == "west bengal") &
    (apy_2024_demo["District "].str.strip().str.lower() == "bankura") &
    (apy_2024_demo["Crop"].str.strip().str.lower() == "wheat")
].copy()

apy_25 = apy_2025_demo[
    (apy_2025_demo["State"].str.strip().str.lower() == "west bengal") &
    (apy_2025_demo["District "].str.strip().str.lower() == "bankura") &
    (apy_2025_demo["Crop"].str.strip().str.lower() == "wheat")
].copy()

print("2024 Demo APY:")
display(apy_24)

print("\n2025 Demo APY:")
display(apy_25)

2024 Demo APY:


,State,District,Crop,Crop_Year,Season,Area,Production,Yield,Demo_Year
18284,West Bengal,BANKURA,Wheat,2018,Rabi,2254.0,5788.0,2.57,2024



2025 Demo APY:


,State,District,Crop,Crop_Year,Season,Area,Production,Yield,Demo_Year
19246,West Bengal,BANKURA,Wheat,2019,Rabi,1986.0,3678.0,1.85,2025


In [19]:
print(apy_24.columns.tolist())
print(apy_25.columns.tolist())

['State', 'District ', 'Crop', 'Crop_Year', 'Season', 'Area ', 'Production', 'Yield', 'Demo_Year']
['State', 'District ', 'Crop', 'Crop_Year', 'Season', 'Area ', 'Production', 'Yield', 'Demo_Year']


In [20]:
print("master_df columns:")
print(master_df.columns.tolist())

master_df columns:
['State', 'District Name', 'Market Name', 'Commodity', 'Price Date', 'Min Price (Rs./Quintal)', 'Max Price (Rs./Quintal)', 'Modal Price (Rs./Quintal)', 'Year', 'Month', 'Month_Name', 'Quarter', 'Day_of_Week', 'Day_Name', 'Day_of_Year', 'Price_Season', 'Crop_Season']


In [21]:
apy_24.columns = apy_24.columns.str.strip()
apy_25.columns = apy_25.columns.str.strip()

print(apy_24.columns.tolist())
print(apy_25.columns.tolist())

['State', 'District', 'Crop', 'Crop_Year', 'Season', 'Area', 'Production', 'Yield', 'Demo_Year']
['State', 'District', 'Crop', 'Crop_Year', 'Season', 'Area', 'Production', 'Yield', 'Demo_Year']


In [22]:
master_df["Area"] = master_df["Year"].map({
    2024: apy_24.iloc[0]["Area"],
    2025: apy_25.iloc[0]["Area"]
})

master_df["Production"] = master_df["Year"].map({
    2024: apy_24.iloc[0]["Production"],
    2025: apy_25.iloc[0]["Production"]
})

master_df["Yield"] = master_df["Year"].map({
    2024: apy_24.iloc[0]["Yield"],
    2025: apy_25.iloc[0]["Yield"]
})

print(master_df[["Year", "Area", "Production", "Yield"]].head())

   Year    Area  Production  Yield
0  2025  1986.0      3678.0   1.85
1  2025  1986.0      3678.0   1.85
2  2024  2254.0      5788.0   2.57
3  2025  1986.0      3678.0   1.85
4  2025  1986.0      3678.0   1.85


In [23]:
import requests

print("requests imported successfully")

latitude = 23.2324
longitude = 87.0784

print("Latitude:", latitude)
print("Longitude:", longitude)

requests imported successfully
Latitude: 23.2324
Longitude: 87.0784


In [24]:
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": latitude,
    "longitude": longitude,
    "start_date": "2024-08-15",
    "end_date": "2025-08-14",
    "daily": "precipitation_sum,temperature_2m_max,temperature_2m_min",
    "timezone": "Asia/Kolkata"
}

response = requests.get(url, params=params)

print("Status code:", response.status_code)

Status code: 200


In [25]:
weather_data = response.json()

weather_df = pd.DataFrame({
    "Price Date": pd.to_datetime(weather_data["daily"]["time"]),
    "Rainfall_mm": weather_data["daily"]["precipitation_sum"],
    "Max_Temp_C": weather_data["daily"]["temperature_2m_max"],
    "Min_Temp_C": weather_data["daily"]["temperature_2m_min"]
})

print(weather_df.shape)
print(weather_df.head())

(365, 4)
  Price Date  Rainfall_mm  Max_Temp_C  Min_Temp_C
0 2024-08-15          4.6        32.8        25.2
1 2024-08-16         10.0        32.8        25.0
2 2024-08-17         10.4        32.2        24.6
3 2024-08-18          8.4        32.0        25.0
4 2024-08-19         12.0        30.8        25.1


In [26]:
master_df = master_df.merge(
    weather_df,
    on="Price Date",
    how="left"
)

print(master_df.shape)
print(master_df[[
    "Price Date",
    "Modal Price (Rs./Quintal)",
    "Rainfall_mm",
    "Max_Temp_C",
    "Min_Temp_C"
]].head())

(364, 23)
  Price Date  Modal Price (Rs./Quintal)  Rainfall_mm  Max_Temp_C  Min_Temp_C
0 2025-04-01                     2900.0          0.0        38.4        23.8
1 2025-08-01                     3000.0         23.0        32.7        25.3
2 2024-12-01                     2700.0          0.1        27.1        17.9
3 2025-02-01                     2800.0          0.0        29.0        17.0
4 2025-01-01                     2850.0          0.0        22.6        12.2


In [27]:
print(master_df[[
    "Rainfall_mm",
    "Max_Temp_C",
    "Min_Temp_C"
]].isna().sum())


Rainfall_mm    0
Max_Temp_C     0
Min_Temp_C     0
dtype: int64


In [28]:
master_df["Lag_1"] = master_df["Modal Price (Rs./Quintal)"].shift(1)
master_df["Lag_7"] = master_df["Modal Price (Rs./Quintal)"].shift(7)
master_df["Lag_14"] = master_df["Modal Price (Rs./Quintal)"].shift(14)
master_df["Lag_30"] = master_df["Modal Price (Rs./Quintal)"].shift(30)

print(master_df[[
    "Price Date",
    "Modal Price (Rs./Quintal)",
    "Lag_1",
    "Lag_7",
    "Lag_14",
    "Lag_30"
]].head(35))

   Price Date  Modal Price (Rs./Quintal)   Lag_1   Lag_7  Lag_14  Lag_30
0  2025-04-01                     2900.0     NaN     NaN     NaN     NaN
1  2025-08-01                     3000.0  2900.0     NaN     NaN     NaN
2  2024-12-01                     2700.0  3000.0     NaN     NaN     NaN
3  2025-02-01                     2800.0  2700.0     NaN     NaN     NaN
4  2025-01-01                     2850.0  2800.0     NaN     NaN     NaN
5  2025-07-01                     3000.0  2850.0     NaN     NaN     NaN
6  2025-06-01                     3000.0  3000.0     NaN     NaN     NaN
7  2025-03-01                     2800.0  3000.0  2900.0     NaN     NaN
8  2025-05-01                     3000.0  2800.0  3000.0     NaN     NaN
9  2024-11-01                     2800.0  3000.0  2700.0     NaN     NaN
10 2024-10-01                     2800.0  2800.0  2800.0     NaN     NaN
11 2024-09-01                     2800.0  2800.0  2850.0     NaN     NaN
12 2025-04-02                     3000.0  2800.0  3

In [29]:
price_col = "Modal Price (Rs./Quintal)"

master_df["Rolling_Mean_7"] = (
    master_df[price_col].shift(1).rolling(7).mean()
)

master_df["Rolling_Mean_14"] = (
    master_df[price_col].shift(1).rolling(14).mean()
)

master_df["Rolling_Mean_30"] = (
    master_df[price_col].shift(1).rolling(30).mean()
)

print(master_df[[
    "Price Date",
    price_col,
    "Rolling_Mean_7",
    "Rolling_Mean_14",
    "Rolling_Mean_30"
]].tail())

    Price Date  Modal Price (Rs./Quintal)  Rolling_Mean_7  Rolling_Mean_14  \
359 2025-01-31                     2800.0     2842.857143      2864.285714   
360 2025-07-31                     3000.0     2828.571429      2864.285714   
361 2025-03-31                     2900.0     2828.571429      2864.285714   
362 2025-05-31                     3000.0     2842.857143      2871.428571   
363 2024-10-31                     2800.0     2871.428571      2885.714286   

     Rolling_Mean_30  
359      2880.000000  
360      2873.333333  
361      2876.666667  
362      2873.333333  
363      2880.000000  


In [30]:
print(master_df[[
    "Lag_1",
    "Lag_7",
    "Lag_14",
    "Lag_30",
    "Rolling_Mean_7",
    "Rolling_Mean_14",
    "Rolling_Mean_30"
]].isna().sum())

Lag_1               1
Lag_7               7
Lag_14             14
Lag_30             30
Rolling_Mean_7      7
Rolling_Mean_14    14
Rolling_Mean_30    30
dtype: int64


In [31]:
feature_cols = [
    "Lag_1",
    "Lag_7",
    "Lag_14",
    "Lag_30",
    "Rolling_Mean_7",
    "Rolling_Mean_14",
    "Rolling_Mean_30"
]

master_df = master_df.dropna(subset=feature_cols).reset_index(drop=True)

print("New shape:", master_df.shape)
print("Remaining missing values:")
print(master_df[feature_cols].isna().sum().sum())


New shape: (334, 30)
Remaining missing values:
0


In [32]:
feature_cols = [
    "Lag_1",
    "Lag_7",
    "Lag_14",
    "Lag_30",
    "Rolling_Mean_7",
    "Rolling_Mean_14",
    "Rolling_Mean_30",
    "Rainfall_mm",
    "Max_Temp_C",
    "Min_Temp_C",
    "Area",
    "Production",
    "Yield",
    "Month",
    "Quarter",
    "Day_of_Week",
    "Day_of_Year"
]

X = master_df[feature_cols]
y = master_df["Modal Price (Rs./Quintal)"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Features:", X.columns.tolist())

X shape: (334, 17)
y shape: (334,)
Features: ['Lag_1', 'Lag_7', 'Lag_14', 'Lag_30', 'Rolling_Mean_7', 'Rolling_Mean_14', 'Rolling_Mean_30', 'Rainfall_mm', 'Max_Temp_C', 'Min_Temp_C', 'Area', 'Production', 'Yield', 'Month', 'Quarter', 'Day_of_Week', 'Day_of_Year']


In [33]:
split_index = int(len(master_df) * 0.8)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

print("Training period:")
print(master_df["Price Date"].iloc[0], "to", master_df["Price Date"].iloc[split_index - 1])

print("\nTesting period:")
print(master_df["Price Date"].iloc[split_index], "to", master_df["Price Date"].iloc[-1])

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))

Training period:
2025-06-03 00:00:00 to 2024-11-25 00:00:00

Testing period:
2024-10-25 00:00:00 to 2024-10-31 00:00:00

Training samples: 267
Testing samples: 67


In [34]:
scale_cols = [    "Lag_1",    "Lag_7",    "Lag_14",    "Lag_30", "Rolling_Mean_7","Rolling_Mean_14",
              "Rolling_Mean_30","Rainfall_mm", "Max_Temp_C", "Min_Temp_C",  "Area",  "Production",
    "Yield"
]


In [35]:
preprocessor = ColumnTransformer(
    transformers=[
        ("scaler", StandardScaler(), scale_cols)
    ],
    remainder="passthrough"
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
   
])

In [36]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    AdaBoostRegressor,
    RandomForestRegressor,
    VotingRegressor
)
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor

models = {
    "Dummy Baseline": DummyRegressor(strategy="mean"),

    "Linear Regression": LinearRegression(),

    "Ridge Regression": Ridge(alpha=1.0),

    "KNN Regressor": KNeighborsRegressor(n_neighbors=5),

    "SVR": SVR(kernel="rbf"),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        random_state=42
    ),

    "Hist Gradient Boosting": HistGradientBoostingRegressor(
        max_iter=100,
        random_state=42
    ),

    "AdaBoost": AdaBoostRegressor(
        n_estimators=100,
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )
}

print("Total models:", len(models))

Total models: 11


In [37]:
from sklearn.base import clone

model_pipelines = {
    name: Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", model)
    ])
    for name, model in models.items()
}

In [38]:
import pandas as pd
import numpy as np

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

results = []
trained_models = {}

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", clone(model))
    ])

    # Train
    pipeline.fit(X_train, y_train)

    # Predict
    y_pred = pipeline.predict(X_test)

    # Evaluate
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2 Score": r2
    })

    trained_models[name] = pipeline

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("MAE").reset_index(drop=True)

print(results_df.round(3))

                     Model     MAE    RMSE  R2 Score
0              Extra Trees   7.733  13.898     0.977
1            KNN Regressor   8.657  17.812     0.962
2        Gradient Boosting   9.031  15.994     0.969
3                  XGBoost  10.110  17.704     0.962
4   Hist Gradient Boosting  10.151  15.554     0.971
5            Random Forest  11.514  18.096     0.961
6                 AdaBoost  14.945  17.755     0.962
7         Ridge Regression  20.024  26.155     0.918
8        Linear Regression  20.140  26.206     0.917
9                      SVR  66.016  86.205     0.105
10          Dummy Baseline  87.130  91.449    -0.007


In [39]:

price_col = "Modal Price (Rs./Quintal)"

# Find rows where the modal price differs from the previous row
price_changes = master_df.loc[
    master_df[price_col].ne(master_df[price_col].shift()),
    ["Price Date", price_col]
]

print(price_changes.to_string(index=False))


Price Date  Modal Price (Rs./Quintal)
2025-06-03                     3000.0
2025-03-03                     2800.0
2025-05-03                     3000.0
2024-11-03                     2800.0
2025-04-04                     3000.0
2024-12-04                     2850.0
2025-02-04                     2800.0
2025-07-04                     3000.0
2025-03-04                     2800.0
2025-05-04                     3000.0
2024-11-04                     2800.0
2024-09-04                     2850.0
2025-04-05                     3000.0
2024-12-05                     2800.0
2025-07-05                     3000.0
2025-03-05                     2800.0
2025-05-05                     3000.0
2024-11-05                     2800.0
2024-09-05                     2850.0
2025-04-06                     3000.0
2024-12-06                     2800.0
2025-07-06                     3000.0
2025-03-06                     2800.0
2025-05-06                     3000.0
2024-11-06                     2800.0
2025-04-07  

In [40]:
price_col = "Modal Price (Rs./Quintal)"

# Check the number of observations and distinct prices by month
master_df["Month_Period"] = master_df["Price Date"].dt.to_period("M")

print(
    master_df.groupby("Month_Period")[price_col]
    .agg(["count", "nunique", "min", "max"])
    .to_string()
)

              count  nunique     min     max
Month_Period                                
2024-08          17        2  2800.0  2850.0
2024-09          28        3  2800.0  2860.0
2024-10          29        1  2800.0  2800.0
2024-11          28        2  2800.0  2850.0
2024-12          28        2  2800.0  2850.0
2025-01          28        2  2800.0  2850.0
2025-02          25        2  2800.0  2850.0
2025-03          28        5  2800.0  3000.0
2025-04          27        1  3000.0  3000.0
2025-05          29        1  3000.0  3000.0
2025-06          28        1  3000.0  3000.0
2025-07          28        1  3000.0  3000.0
2025-08          11        1  3000.0  3000.0


In [41]:
# Restrict the diagnostic evaluation to dates through April 2025
eval_df = master_df.loc[
    master_df["Price Date"] <= "2025-04-30"
].copy()

print("Observations:", len(eval_df))
print("Date range:", eval_df["Price Date"].min(),
      "to", eval_df["Price Date"].max())
print("Unique prices:",
      eval_df["Modal Price (Rs./Quintal)"].nunique())

Observations: 238
Date range: 2024-08-15 00:00:00 to 2025-04-30 00:00:00
Unique prices: 6


In [42]:
eval_X = X.loc[eval_df.index]
eval_y = y.loc[eval_df.index]

split_idx = int(len(eval_X) * 0.8)

X_train_diag = eval_X.iloc[:split_idx]
X_test_diag = eval_X.iloc[split_idx:]

y_train_diag = eval_y.iloc[:split_idx]
y_test_diag = eval_y.iloc[split_idx:]

print("Train dates:",
      eval_df.loc[X_train_diag.index, "Price Date"].min(),
      "to",
      eval_df.loc[X_train_diag.index, "Price Date"].max())

print("Test dates:",
      eval_df.loc[X_test_diag.index, "Price Date"].min(),
      "to",
      eval_df.loc[X_test_diag.index, "Price Date"].max())

print("Unique test prices:", y_test_diag.nunique())

Train dates: 2024-08-15 00:00:00 to 2025-04-25 00:00:00
Test dates: 2024-08-26 00:00:00 to 2025-04-30 00:00:00
Unique test prices: 5


In [43]:
print("eval_df dates sorted:",
      eval_df["Price Date"].is_monotonic_increasing)

print("eval_X index matches eval_df:",
      eval_X.index.equals(eval_df.index))

print("eval_y index matches eval_df:",
      eval_y.index.equals(eval_df.index))

print("First 10 eval_X dates:")
print(eval_df.loc[eval_X.index, "Price Date"].head(10).to_list())

eval_df dates sorted: False
eval_X index matches eval_df: True
eval_y index matches eval_df: True
First 10 eval_X dates:
[Timestamp('2025-03-03 00:00:00'), Timestamp('2024-11-03 00:00:00'), Timestamp('2024-10-03 00:00:00'), Timestamp('2024-09-03 00:00:00'), Timestamp('2025-04-04 00:00:00'), Timestamp('2024-12-04 00:00:00'), Timestamp('2025-02-04 00:00:00'), Timestamp('2025-01-04 00:00:00'), Timestamp('2025-03-04 00:00:00'), Timestamp('2024-11-04 00:00:00')]


In [44]:
# Sort diagnostic data chronologically
eval_df = eval_df.sort_values("Price Date").copy()

# Keep X and y aligned to the sorted DataFrame
eval_X = X.loc[eval_df.index].copy()
eval_y = y.loc[eval_df.index].copy()

print("Dates sorted:",
      eval_df["Price Date"].is_monotonic_increasing)

print("X aligned:", eval_X.index.equals(eval_df.index))
print("y aligned:", eval_y.index.equals(eval_df.index))

Dates sorted: True
X aligned: True
y aligned: True


In [45]:
split_idx = int(len(eval_X) * 0.8)

X_train_diag = eval_X.iloc[:split_idx].copy()
X_test_diag = eval_X.iloc[split_idx:].copy()

y_train_diag = eval_y.iloc[:split_idx].copy()
y_test_diag = eval_y.iloc[split_idx:].copy()

print("Train dates:",
      eval_df.loc[X_train_diag.index, "Price Date"].min(),
      "to",
      eval_df.loc[X_train_diag.index, "Price Date"].max())

print("Test dates:",
      eval_df.loc[X_test_diag.index, "Price Date"].min(),
      "to",
      eval_df.loc[X_test_diag.index, "Price Date"].max())

Train dates: 2024-08-15 00:00:00 to 2025-03-10 00:00:00
Test dates: 2025-03-11 00:00:00 to 2025-04-30 00:00:00


In [46]:
market_coverage = (
    df.groupby(["State", "District Name"])
      .agg(
          Markets=("Market Name", "nunique"),
          Commodities=("Commodity", "nunique"),
          Observations=("Price Date", "size")
      )
      .reset_index()
      .sort_values(
          ["Markets", "Observations"],
          ascending=False
      )
)

print(market_coverage.head(15).to_string(index=False))

         State      District Name  Markets  Commodities  Observations
        Kerala           Kottayam       23           10         13372
        Punjab           Ludhiana       22           13          7206
        Kerala     Pathanamthitta       22            5          2848
       Gujarat        Sabarkantha       21            9          5453
        Punjab          Jalandhar       18           12          8067
Madhya Pradesh              Sagar       16           19          8629
       Gujarat         Banaskanth       16           16          8162
        Kerala             Kollam       16            9          7468
        Kerala            Palakad       15           11         16090
        Kerala           Thirssur       15            9          9654
       Gujarat              Surat       15           18          4979
        Kerala Thiruvananthapuram       14           10         14491
        Kerala          Alappuzha       14            9         13224
        Kerala Kozhi

In [47]:
bankura_df = df[
    (df["State"] == "West Bengal") &
    (df["District Name"] == "Bankura")
].copy()

print("Shape:", bankura_df.shape)
print("Markets:", bankura_df["Market Name"].nunique())
print("Commodities:", bankura_df["Commodity"].nunique())

print("\nMarkets:")
print(sorted(bankura_df["Market Name"].unique()))

print("\nCommodities:")
print(sorted(bankura_df["Commodity"].unique()))

Shape: (3486, 11)
Markets: 4
Commodities: 4

Markets:
['Bankura Sadar', 'Bishnupur(Bankura)', 'Indus(Bankura Sadar)', 'Khatra']

Commodities:
['Brinjal', 'Green Chilli', 'Mustard', 'Wheat']
